In [0]:
import datetime
import pandas as pd
import numpy as np

In [0]:
ACTUALS_PATH     = '/dbfs/mnt/thesis/output_data/processed_data.csv'
BENCHMARK_PATH = '/dbfs/mnt/thesis/predictions/benchmark/'
PREDICTIONS_PATH = '/dbfs/mnt/thesis/predictions/lgbm/ensemble_rolling_periodic_retune/'
TODAY     = pd.to_datetime('2010-08-09')

In [0]:
actual_df = pd.read_csv(ACTUALS_PATH,parse_dates=['DATETIME'])
actual_df = actual_df[
    (actual_df.DATETIME >= (TODAY - datetime.timedelta(days=3)))
    & (actual_df.DATETIME < (TODAY))
]
actual_df = actual_df[actual_df['LOCATION'] < 48]
actual_df.rename(columns={'VALUE':'actual'},inplace=True)
print(actual_df['DATETIME'].min(),actual_df['DATETIME'].max())

2010-08-06 00:00:00 2010-08-08 23:55:00


In [0]:
bm_df = pd.read_csv(BENCHMARK_PATH + f'predictions_{TODAY.date()}.csv',parse_dates=['DATETIME'])
bm_df = bm_df[bm_df['LOCATION'] < 48]
bm_df.drop(columns="VALUE",inplace=True)
bm_df.rename(columns={'bm_1d_prediction':'benchmark'},inplace=True)
print(bm_df['DATETIME'].min(),bm_df['DATETIME'].max())

2010-08-09 00:00:00 2010-08-09 23:55:00


In [0]:
pred_df = pd.read_csv(PREDICTIONS_PATH + f'predictions_{TODAY.date()}.csv',parse_dates=['DATETIME'])
pred_df = pred_df[pred_df['LOCATION'] < 48]
pred_df.rename(columns={'PREDICTED':'prediction'},inplace=True)
print(pred_df['DATETIME'].min(),pred_df['DATETIME'].max())

2010-08-09 00:00:00 2010-08-09 23:55:00


In [0]:
eval_df = bm_df.merge(
    pred_df, on=['DATETIME','LOCATION'], how='inner', validate='one_to_one'
)
result_df = pd.concat([actual_df,eval_df]).reset_index(drop=True)
result_df.to_csv('/dbfs/mnt/thesis/mlops/lgbm/timeseries.csv',index=False)